# 04 — DistilBERT fine-tuning (Colab GPU)

**Run this on Colab Pro with an L4 or A100 runtime.** Local CPU is too slow for full training.

Expected wall time: ~10 minutes for 3 epochs on L4 with ~10k headlines.
Expected accuracy: 80–82%.

Setup steps:
1. Clone the repo, mount Drive (so checkpoints survive disconnects).
2. Install transformers stack.
3. Run `src.models.transformer` with `distilbert-base-uncased`.
4. Save the final model to Drive and copy back into `models/`.

In [ ]:
from google.colab import drive  # only on Colab
drive.mount('/content/drive')

In [ ]:
# Clone repo (replace with your team's URL)
# !git clone https://github.com/<your-team>/cis-4190-project.git
# %cd cis-4190-project

In [ ]:
%pip install -q transformers==4.45.* datasets accelerate evaluate scikit-learn

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Smoke test on 200 rows / 1 epoch

In [ ]:
!python -m src.models.transformer \
    --model distilbert-base-uncased \
    --split random \
    --epochs 1 \
    --max-samples 200 \
    --output-dir /content/drive/MyDrive/cis5190/runs/distilbert_smoke

## Full training run

In [ ]:
!python -m src.models.transformer \
    --model distilbert-base-uncased \
    --split random \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-length 128 \
    --output-dir /content/drive/MyDrive/cis5190/runs/distilbert_v1

## Hyperparameter sweep (lr × epochs)

In [ ]:
for lr in [2e-5, 3e-5, 5e-5]:
    for ep in [2, 3]:
        run_name = f'distilbert_lr{lr:.0e}_e{ep}'
        !python -m src.models.transformer \
            --model distilbert-base-uncased \
            --split random --epochs {ep} --lr {lr} \
            --output-dir /content/drive/MyDrive/cis5190/runs/{run_name}

## Train best config on the temporal split too (for the analysis)

In [ ]:
!python -m src.models.transformer \
    --model distilbert-base-uncased \
    --split temporal \
    --epochs 3 --batch-size 32 --lr 2e-5 \
    --output-dir /content/drive/MyDrive/cis5190/runs/distilbert_temporal

## Copy best model into the repo for submission

In [ ]:
!mkdir -p models/distilbert_final
!cp -r /content/drive/MyDrive/cis5190/runs/distilbert_v1/final/* models/distilbert_final/
!ls -la models/distilbert_final